# Public source archive

Recovered historical code, prepared for public review. AI assisted the original coding and this archive preparation; team projects retain team attribution in the project README.

Outputs, notebook attachments, personal metadata, and embedded organizer datasets have been removed. Narrative cells retain headings only. Supply authorized input files before running data-dependent notebooks. This archive does not claim that manual checks were completed or that the code was independently developed.


# Early-Career Talent: Employment Offering and Recruitment Messaging


## 1. Load data and verify scope


In [ ]:
import json, os
EXCLUDED_JOB_IDS = json.loads(os.environ.get("EXCLUDED_JOB_IDS", "[]"))
from pathlib import Path
import os, re, json, hashlib, platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
from html import escape
DATA = Path(os.environ.get('HACKATHON_DATA_DIR', '/content' if Path('/content').exists() else '.')).resolve()
DATA.mkdir(parents=True, exist_ok=True)
OUT = DATA / 'hackathon_results'
OUT.mkdir(exist_ok=True)
COMPANIES = ['Accenture', 'Booz Allen Hamilton', 'Capgemini']
FOCAL = 'Accenture'
assert FOCAL in COMPANIES
NAMES = ['organizations.csv', 'job_postings.csv', 'reviews.csv', 'review_job_role_links.csv']

def candidates(name):
    return sorted((p for p in DATA.glob('*.csv') if p.name == name or p.name.endswith('-' + name)))
if any((not candidates(name) for name in NAMES)):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f'Place the four original CSV files in {DATA}, then rerun.') from exc
    print('Upload the four original CSV files together. A saved manual_checks.csv may also be uploaded.')
    uploaded = files.upload()
    for filename, content in uploaded.items():
        (DATA / Path(filename).name).write_bytes(content)
paths = {}
for name in NAMES:
    found = candidates(name)
    if len(found) != 1:
        raise ValueError(f'Found {len(found)} copies of {name}. Keep one original copy in the data folder.')
    paths[name] = found[0]
raw_tables = {name: pd.read_csv(path, dtype=str, encoding='utf-8-sig').fillna('') for name, path in paths.items()}
org = raw_tables['organizations.csv']
selected_org = org[org['organization_name'].isin(COMPANIES)].copy()
assert set(selected_org['organization_name']) == set(COMPANIES), 'An employer is missing.'
org_ids = set(selected_org['organization_id'])
duplicate_log = []

def select_unique(name, id_col):
    frame = raw_tables[name]
    frame = frame[frame['organization_id'].isin(org_ids)].copy()
    duplicate_log.append({'source': name, 'identical_rows_removed': int(frame.duplicated().sum())})
    frame = frame.drop_duplicates()
    if frame[id_col].eq('').any() or frame[id_col].duplicated().any():
        raise ValueError(f'{name} contains missing or conflicting IDs. Inspect them before continuing.')
    return frame
J = select_unique('job_postings.csv', 'job_id')
R = select_unique('reviews.csv', 'review_id')
L = raw_tables['review_job_role_links.csv']
L = L[L['organization_id'].isin(org_ids)].drop_duplicates().copy()
assert L['job_id'].isin(J['job_id']).all(), 'A link refers to an unavailable posting.'
assert L['review_id'].isin(R['review_id']).all(), 'A link refers to an unavailable review.'
J['attribution_pending'] = J['job_id'].isin(EXCLUDED_JOB_IDS)
J_main = J[~J['attribution_pending']].copy()

def scope_table():
    rows = []
    for company in COMPANIES:
        a = J[J.organization_name.eq(company)]
        b = R[R.organization_name.eq(company)]
        rows.append({'company': company, 'raw_selected_postings': len(a), 'main_comparison_postings': int((~a.attribution_pending).sum()), 'reviews': len(b), 'distinct_posting_titles': a.title.nunique(), 'reviews_missing_job_title': int(b.reviewer_job_title.eq('').sum()), 'repeated_posting_text_hash_rows': int(a.loc[a.text_sha256.ne(''), 'text_sha256'].duplicated().sum()), 'repeated_review_text_hash_rows': int(b.loc[b.text_sha256.ne(''), 'text_sha256'].duplicated().sum())})
    return pd.DataFrame(rows).set_index('company')
scope = scope_table()
display(scope)
scope.to_csv(OUT / 'sample_scope.csv')
pd.DataFrame(duplicate_log).to_csv(OUT / 'duplicate_row_log.csv', index=False)
exclusions = J.loc[J.attribution_pending, ['organization_name', 'job_id', 'title']].copy()
exclusions['reason'] = 'Configure an exclusion reason from your authorized review.'
exclusions.to_csv(OUT / 'posting_exclusions.csv', index=False)
manifest = {'research_question': 'How should an organization improve its employment offering and recruitment messaging to attract early-career talent, based on its job postings, employee reviews, and comparable employers?', 'focal_employer': FOCAL, 'companies': COMPANIES, 'input_sha256': {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in paths.items()}, 'python': platform.python_version(), 'packages': {name: metadata.version(name) for name in ['pandas', 'numpy', 'matplotlib', 'IPython', 'ipywidgets']}, 'rules_version': 'original_uploaded_dictionary_preserved_english_submission_v1', 'exclusion': {}, 'notes': 'Historical snapshots. Role links are not confirmed same openings. Advertising baseline documented retrospectively.'}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Results folder: {OUT}')


## 2. Establish the posting-only baseline


In [ ]:
THEMES = {
    'Career development': r'\btrain[A-Za-z_]*\b|\blearn[A-Za-z_]*\b|\bmentor[A-Za-z_]*\b|\bcoach[A-Za-z_]*\b|\bonboard[A-Za-z_]*\b|\bupskill[A-Za-z_]*\b|\bcareer\b|\bpromot[A-Za-z_]*\b|\bprogression\b|\btuition\b|\bcourses?\b',
    'Flexibility & workload': r'\bhybrid\b|\bremote\b|\bflexib[A-Za-z_]*\b|\bwork.from.home\b|\bwfh\b|\bwork.life\b|\bworkload\b|\bovertime\b|\blong hours\b|\brotational shifts?\b|\bburnout\b|\bburn.out\b',
    'Compensation & benefits': r'\bsalar[A-Za-z_]*\b|\bpay\b|\bcompensation\b|\bbenefits?\b|\bbonus[A-Za-z_]*\b|\binsurance\b|\bretirement\b|\b401\(?k\)?\b|\bpension\b|\bpaid leave\b|\bpaid time off\b|\bhikes?\b'
}
TITLE_SIGNAL = r'\bintern(?:s|ship|ships)?\b|\bjunior\b|\bgraduates?\b|\bentry[ -]level\b|\btrainees?\b|\bapprentices?\b'
BODY_SIGNAL = TITLE_SIGNAL + r'|\bfreshers?\b|\bearly[ -]career\b|\bfirst job\b|\bnew joiners?\b'

def plain(s):
    return ' '.join(str(s).split())

def theme_flag(s, pattern):
    cleaned = re.sub(r'\b(?:machine|deep)[ -]learning\b', '', plain(s), flags=re.I)
    return bool(re.search(pattern, cleaned, flags=re.I))

for frame in [J, J_main]:
    frame['text'] = frame.description
    frame['early_title_signal'] = frame.title.str.contains(TITLE_SIGNAL, case=False, regex=True)
    frame['early_body_signal'] = frame.description.str.contains(BODY_SIGNAL, case=False, regex=True)
    frame['early_signal'] = frame[['early_title_signal', 'early_body_signal']].any(axis=1)
    for theme, pattern in THEMES.items():
        frame[theme] = frame.text.map(lambda value, p=pattern: theme_flag(value, p))

def theme_summary(frame):
    rows = []
    for company in COMPANIES:
        group = frame[frame.organization_name.eq(company)]
        for theme in THEMES:
            n, count = len(group), int(group[theme].sum())
            rows.append({'company':company, 'theme':theme, 'n':n,
                         'mentions':count, 'percent':100*count/n if n else np.nan})
    return pd.DataFrame(rows)

ad_baseline = theme_summary(J_main)
def formatted_summary(table):
    temp = table.copy()
    temp['count/n (%)'] = temp.apply(
        lambda x: f"{int(x['mentions'])}/{int(x['n'])} ({x['percent']:.1f}%)" if x['n'] else 'N/A', axis=1)
    return temp.pivot(index='company', columns='theme', values='count/n (%)').reindex(COMPANIES)

display(formatted_summary(ad_baseline))
ad_baseline.to_csv(OUT/'postings_only_baseline.csv', index=False)
print('Text-mention baseline only. Career-stage signals require eligibility and context checks; other postings provide organizational context.')
(OUT / 'theme_definitions.json').write_text(json.dumps({'themes': THEMES, 'title_signal': TITLE_SIGNAL, 'broad_body_signal': BODY_SIGNAL}, indent=2), encoding='utf-8')

### Posting-only interpretation


## 3. Add employee reviews and build an evidence index


In [ ]:
FIELDS = ['summary', 'pros', 'cons', 'advice_to_management']
R['text'] = R[FIELDS].agg('\n'.join, axis=1)
R['early_title_signal'] = R.reviewer_job_title.str.contains(TITLE_SIGNAL, case=False, regex=True)
R['early_body_signal'] = R.text.str.contains(BODY_SIGNAL, case=False, regex=True)
R['early_signal'] = R[['early_title_signal', 'early_body_signal']].any(axis=1)
for theme, pattern in THEMES.items():
    R[theme] = R.text.map(lambda value, p=pattern: theme_flag(value, p))
review_summary = theme_summary(R)
early_review_summary = theme_summary(R[R.early_signal])
display(formatted_summary(review_summary))

evidence_rows = []
for source, frame, identifier, title_col, fields in [
    ('posting', J, 'job_id', 'title', ['description']),
    ('review', R, 'review_id', 'reviewer_job_title', FIELDS)
]:
    for _, row in frame.iterrows():
        for field in fields:
            value = plain(row[field])
            for theme, pattern in THEMES.items():
                if not theme_flag(value, pattern):
                    continue
                evidence_rows.append({
                    'source':source, 'record_id':row[identifier],
                    'company':row.organization_name, 'title':row[title_col],
                    'early_signal':row.early_signal, 'theme':theme,
                    'field':field, 'full_field_text':value,
                    'attribution_pending':bool(row.get('attribution_pending', False)),
                    'human_status':'pending'
                })
evidence = pd.DataFrame(evidence_rows)
evidence.to_csv(OUT/'evidence_index.csv', index=False)

def show_evidence(company=FOCAL, source='review', theme='Career development', early_only=False, limit=5):
    subset = evidence[(evidence.company.eq(company)) & (evidence.source.eq(source)) & (evidence.theme.eq(theme))]
    if early_only:
        subset = subset[subset.early_signal]
    print(f'Matching field passages: {len(subset)}; distinct records: {subset.record_id.nunique()}. Showing the first {limit}.')
    html = []
    for _, row in subset.sort_values(['record_id','field']).head(limit).iterrows():
        heading = f'{row.company} | {row.source} {row.record_id} | {row.title} | {row.field}'
        html.append(f'<details><summary>{escape(heading)}</summary><p>{escape(row.full_field_text)}</p></details>')
    display(HTML('<br>'.join(html)))

show_evidence()

review_summary.to_csv(OUT / 'review_theme_summary.csv', index=False)
early_review_summary.to_csv(OUT / 'broad_signal_review_theme_summary.csv', index=False)

### What changes after adding reviews?


## 4. Visualize source-specific results


In [ ]:
plt.rcParams.update({'figure.dpi':120, 'font.size':10})
COLORS = ['#3659A2', '#26A69A', '#F4A340']

# Figure 1: Count postings and reviews separately.
counts = pd.DataFrame({
    'Postings (main comparison)': J_main.groupby('organization_name').size(),
    'Reviews': R.groupby('organization_name').size()
}).reindex(COMPANIES)
ax = counts.plot.bar(figsize=(9,4), color=COLORS[:2], rot=0)
ax.set(title='1. Source-specific sample sizes', xlabel='', ylabel='Distinct records')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'01_sample_sizes.png'); plt.show()

def theme_plot(table, title, filename):
    wide = table.pivot(index='company', columns='theme', values='percent').reindex(index=COMPANIES, columns=list(THEMES))
    ax = wide.plot.bar(figsize=(10,4), color=COLORS, rot=0)
    ax.set(title=title, xlabel='', ylabel='Records mentioning theme (%)', ylim=(0,105))
    ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5,-0.13), ncol=3)
    for i, company in enumerate(COMPANIES):
        n = int(table.loc[table.company.eq(company), 'n'].iloc[0])
        ax.text(i, 102, f'n={n}', ha='center', fontsize=8)
    plt.tight_layout(); plt.savefig(OUT/filename, bbox_inches='tight'); plt.show()

theme_plot(ad_baseline, '2. Posting theme mentions — not verified promises', '02_posting_mentions.png')
theme_plot(review_summary, '3. Review theme mentions — not sentiment', '03_review_mentions.png')
theme_plot(early_review_summary, '4. Reviews with early-career signals — eligibility unverified', '04_early_signal_reviews.png')

## 5. Check sensitivity and evidence comparability


In [ ]:
def deduplicate_by_key(frame, column):
    nonempty = frame[frame[column].ne('')].drop_duplicates(['organization_id', column])
    empty = frame[frame[column].eq('')]
    return pd.concat([nonempty, empty], ignore_index=True)

variants = {
    'Main comparison': J_main,
    'Include attribution-pending posting': J,
    'Distinct exact text within employer': deduplicate_by_key(J_main, 'text_sha256'),
    'Distinct opening key within employer': deduplicate_by_key(J_main, 'opening_dedup_key')
}
sensitivity = pd.concat([theme_summary(frame).assign(variant=name)
                         for name, frame in variants.items()], ignore_index=True)
base = ad_baseline[['company', 'theme', 'percent']].rename(columns={'percent': 'main_percent'})
sensitivity = sensitivity.merge(base, on=['company', 'theme'], validate='many_to_one')
sensitivity['change_percentage_points'] = sensitivity['percent'] - sensitivity['main_percent']
display(sensitivity[['variant', 'company', 'theme', 'n', 'mentions', 'percent', 'change_percentage_points']].round(2))
sensitivity.to_csv(OUT / 'posting_sensitivity.csv', index=False)

date_rows = []
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
all_dates = pd.concat([pd.to_datetime(R.review_date, errors='coerce', utc=True),
                       pd.to_datetime(J_main.posted_at, errors='coerce', utc=True)]).dropna()
months = pd.period_range(all_dates.min().tz_localize(None).to_period('M'),
                         all_dates.max().tz_localize(None).to_period('M'), freq='M')
for ax, (source, frame, date_col) in zip(axes, [('Reviews', R, 'review_date'), ('Postings', J_main, 'posted_at')]):
    for company, color in zip(COMPANIES, COLORS):
        dates = pd.to_datetime(frame.loc[frame.organization_name.eq(company), date_col], errors='coerce', utc=True)
        monthly = dates.dt.tz_localize(None).dt.to_period('M').value_counts().reindex(months, fill_value=0)
        ax.plot(range(len(months)), monthly.values, marker='o', label=company, color=color)
        date_rows.append({'source': source, 'company': company, 'dated_records': int(dates.notna().sum()),
                          'missing_or_invalid_dates': int(dates.isna().sum()),
                          'earliest_date': str(dates.min()), 'latest_date': str(dates.max())})
    ax.set(title=f'{source} by source month', ylabel='Records')
axes[-1].set_xticks(range(len(months)))
axes[-1].set_xticklabels([str(month) for month in months], rotation=45, ha='right')
axes[0].legend(fontsize=8)
fig.suptitle('5. Dates represented in the sample (not a business trend)')
fig.tight_layout()
fig.savefig(OUT / '05_source_dates.png', bbox_inches='tight')
plt.show()
date_scope = pd.DataFrame(date_rows)
display(date_scope)
date_scope.to_csv(OUT / 'source_date_scope.csv', index=False)

print('Location labels in the supplied comparison file:', sorted(L.location_match.unique()))
links = L[L.job_id.isin(J_main.job_id)].copy()
links = links.merge(selected_org[['organization_id', 'organization_name']].drop_duplicates(),
                    on='organization_id', validate='many_to_one')
gap = pd.to_numeric(links.posting_minus_review_days, errors='coerce').abs()
rows = []
for company in COMPANIES:
    selected = links.organization_name.eq(company)
    rows.append({'company': company,
                 'unique_same_title_linked_reviews': links.loc[selected, 'review_id'].nunique(),
                 'unique_within_180_days': links.loc[selected & gap.le(180), 'review_id'].nunique(),
                 'unique_strict_linked_reviews': links.loc[selected & gap.le(180) & links.location_match.str.lower().eq('exact_location_text'), 'review_id'].nunique()})
link_scope = pd.DataFrame(rows)
display(link_scope)
link_scope.to_csv(OUT / 'role_link_scope.csv', index=False)
print('Source months describe the sample, not hiring trends. Links do not add independent observations.')

### 6. Interpret focal-employer and peer evidence


In [ ]:
peer_cases = pd.DataFrame(columns=['company', 'posting_ids', 'advertised_arrangement', 'positive_review_ids', 'positive_account', 'concern_review_ids', 'reported_concern', 'interpretation_label', 'question_to_investigate'])
display(peer_cases)
peer_cases.to_csv(OUT / 'peer_evidence_cases.csv', index=False)

def show_record(source, record_id):
    frame, id_col = (J, 'job_id') if source == 'posting' else (R, 'review_id')
    match = frame[frame[id_col].eq(str(record_id))]
    if match.empty:
        raise KeyError(f'No {source} record with ID {record_id}.')
    row = match.iloc[0]
    print(row.organization_name, '|', source, record_id)
    if source == 'posting':
        print(row.title, '|', row.location_text, '|', row.posted_at)
        print(row.description)
    else:
        print(row.reviewer_job_title, '|', row.reviewer_location, '|', row.review_date)
        for field in FIELDS:
            print(f'\n{field}:\n{row[field]}')


## 7. Recommend two priorities for the next recruitment cycle


In [ ]:
recommendations = pd.DataFrame(columns=['priority', 'focal_employer', 'action', 'employment_offering_change', 'recruitment_message_change', 'posting_ids', 'review_ids', 'peer_context', 'interpretation_label', 'reasoning', 'uncertainty', 'future_metric_or_experiment'])
with pd.option_context('display.max_colwidth', 100):
    display(recommendations)
recommendations.to_csv(OUT / 'recommendations.csv', index=False)
interview_questions = ['What training, mentor support, and real project work can a new hire expect in the first 90 days? Can you give a recent example?', 'What are the actual working hours, location requirements, and peak periods for this team? How are project changes communicated?', 'How do pay, benefits, and progression criteria vary by role, location, project, or employee eligibility?']
for i, question in enumerate(interview_questions, 1):
    print(f'{i}. {question}')
(OUT / 'student_interview_questions.txt').write_text('\n'.join((f'{i}. {q}' for i, q in enumerate(interview_questions, 1))), encoding='utf-8')


## 9. Limitations, AI use, and submission status
